# FSG Customer Service Agent

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import pandas as pd

In [3]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "fsg.db"

In [7]:
system_message = """
You are a helpful assistant for an Fencing Supply Group called Chain Link Chuck.
Give charismatic, friendly answers that promote the product. no more than 2 sentences.
Slight creative embelishment is allowed. If you don't know the answer, say so.
"""

In [5]:
# Read in data and save to db

data = pd.read_csv("fencing_products_demo.csv")

with sqlite3.connect(DB) as conn:
    data.to_sql("products", conn, if_exists="replace", index=False)

In [6]:
data.head()

,product_id,product_name,category,material,height_ft,width_ft,color_or_finish,price_usd,stock_status,description
0,FSG-001,Cedar Privacy Fence Panel,Panel,Western Red Cedar,6.0,8.0,Natural Cedar,129.99,In Stock,Premium western red cedar privacy panel ideal ...
1,FSG-002,Cedar Dog-Ear Fence Board,Board,Western Red Cedar,6.0,0.5,Natural Cedar,4.25,In Stock,Traditional dog-ear cedar picket used for clas...
2,FSG-003,Pressure-Treated Pine Fence Board,Board,Pressure Treated Pine,6.0,0.5,Green Treated,3.10,In Stock,Affordable treated pine board designed for lon...
3,FSG-004,Black Aluminum Pool Fence Panel,Panel,Aluminum,5.0,6.0,Black Powder Coat,189.99,In Stock,Decorative aluminum panel commonly used for po...
4,FSG-005,Aluminum Fence Gate 48in,Gate,Aluminum,4.0,4.0,Black Powder Coat,249.00,In Stock,Pre-assembled aluminum gate compatible with al...


In [ ]:
def get_product(product_id: str) -> dict | None:
    """Find a product by product_id. Returns row as dict or None if not found."""
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.execute(
            "SELECT * FROM products WHERE product_id = ?", (product_id,)
        )
        row = cur.fetchone()
    return dict(row) if row else None

In [ ]:
def generate_product_image(product_name: str) -> str:
    """Generate a creative interpretation of the product using DALL-E 3. Returns base64-encoded image."""
    prompt = f"Creative interpretation of this fencing product: {product_name}"
    response = openai.images.generate(
        model="dall-e-3",
        prompt=prompt,
        response_format="b64_json",
        size="1024x1024",
    )
    return response.data[0].b64_json

In [ ]:
def communicate(message: str):
    """Convert message to speech using OpenAI TTS. Returns audio bytes (mp3)."""
    response = openai.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=message,
    )
    return response.content

In [ ]:
def chat(history):
    history = []